In [40]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/mpuu00001/Siformer /content/Siformer
%cd /content/Siformer
s = open('train.py').read()
s = s.replace('+ str(i) + ".pth")', '+ str(i) + ".pth", weights_only=False)')
s = s.replace('_, _, top_val_acc = evaluate_top_k(slr_model, val_loader, device)', 'pass')
open('train.py', 'w').write(s)
!grep -n "weights_only\|evaluate_top_k(slr" train.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path '/content/Siformer' already exists and is not an empty directory.
/content/Siformer
294:                    "out-checkpoints/" + args.experiment_name + "/checkpoint_" + checkpoint_id + "_" + str(i) + ".pth", weights_only=False)


In [41]:
# Ô 2: xem trong thư mục dữ liệu có những file nào (sửa tên thư mục cho đúng)
DATA = '/content/drive/MyDrive/Colab Notebooks/Siformer/WLASL100'
!ls -la "$DATA"

total 642328
dr-x------ 2 root root      4096 Jul 24  2024 .
dr-x------ 4 root root      4096 Jul 24  2024 ..
-r-------- 1 root root 526695314 Jul 24  2024 WLASL100_train_25fps.csv
-r-------- 1 root root 131039549 Mar 13  2024 WLASL100_val_25fps.csv


In [42]:
# Ô 3: đếm số mẫu mỗi lớp trong từng file
import pandas as pd, glob, ast
for f in sorted(glob.glob(DATA + '/*.csv')):
    df = pd.read_csv(f)
    c = df['labels'].value_counts()
    L = df['leftEar_X'].map(lambda s: len(ast.literal_eval(s)))
    print(f.split('/')[-1], '| mẫu:', len(df), '| lớp:', c.size, '| mỗi lớp min/max:', c.min(), c.max(),
          '| frame min/max:', L.min(), L.max(), '| nhãn:', df['labels'].min(), '-', df['labels'].max(),
          '| có neck:', 'neck_X' in df.columns)

WLASL100_train_25fps.csv | mẫu: 3200 | lớp: 100 | mỗi lớp min/max: 32 32 | frame min/max: 204 204 | nhãn: 0 - 99 | có neck: True
WLASL100_val_25fps.csv | mẫu: 800 | lớp: 100 | mỗi lớp min/max: 8 8 | frame min/max: 204 204 | nhãn: 0 - 99 | có neck: True


In [43]:
# Ô 3b: kiểm tra độ dài chuỗi và tên cột
import ast
df = pd.read_csv(DATA + '/WLASL100_train_25fps.csv')
L = df['leftEar_X'].map(lambda s: len(ast.literal_eval(s)))
print('số frame mỗi mẫu: min', L.min(), '| max', L.max())
print('có cột neck:', 'neck_X' in df.columns)
print('nhãn từ', df['labels'].min(), 'đến', df['labels'].max())
print([c for c in df.columns if 'index' in c][:4])

số frame mỗi mẫu: min 204 | max 204
có cột neck: True
nhãn từ 0 đến 99
['indexDIP_left_X', 'indexDIP_left_Y', 'indexDIP_right_X', 'indexDIP_right_Y']


In [44]:
import os
for d in ['datasets', 'normalization', 'augmentations', 'siformer']:
    p = f'/content/Siformer/{d}'
    if os.path.isdir(p) and not os.path.exists(p + '/__init__.py'):
        open(p + '/__init__.py', 'w').close()
        print('đã thêm', p + '/__init__.py')
!python -c "import sys; sys.path.insert(0, '/content/Siformer'); from datasets.czech_slr_dataset import CzechSLRDataset; print('OK: đọc được Dataset của tác giả')"

đã thêm /content/Siformer/datasets/__init__.py
đã thêm /content/Siformer/normalization/__init__.py
đã thêm /content/Siformer/siformer/__init__.py
OK: đọc được Dataset của tác giả


In [45]:
OUT = '/content/drive/MyDrive/DATH_results/repro'
!mkdir -p "$OUT"
%cd $OUT
!PYTHONPATH=/content/Siformer python -m train --experiment_name repro_wlasl100 --num_classes 100 --num_worker 2 \
  --training_set_path "$DATA/WLASL100_train_25fps.csv" \
  --validation_set from-file --validation_set_path "$DATA/WLASL100_val_25fps.csv" \
  --testing_set_path "$DATA/WLASL100_val_25fps.csv" 2>&1 | tee repro_train.log

/content/drive/MyDrive/DATH_results/repro
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:145: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
Cuda is available: True
Feature isolated transformer
self.selected_attn prob
Encoder with input adaptive
self.selected_attn prob
Encoder with input adaptive
self.selected_attn prob
Encoder with input adaptive
Using custom DecoderLayer
Normal decoder
num_enc_layers 3, num_dec_layers 2, patient 1
Starting repro_wlasl100...
Training using /content/drive/MyDrive/Colab Notebooks/Siformer/WLASL100/WLASL100_train_25fps.csv...
Validation using /content/drive/MyDrive/Colab Notebooks/Siformer/WLASL100/WLASL100_val_25fps.csv...


Save checkpoint for [1] as out-checkpoints/repro_wlasl100/checkpoint_v_0.pth
[1] TRAIN  loss: 4.627037389954524 acc: 0.0190625
[1] AV

In [ ]:
import os
ROOT = '/content/data'
OUT = '/content/drive/MyDrive/DATH_results'
for data in ('csv', 'csv_rect_dip', 'csv_rect_full'):
    name = f"siformer_{data}_s0"
    os.makedirs(f"{OUT}/{name}", exist_ok=True); os.chdir(f"{OUT}/{name}")
    CSV = f"{ROOT}/{data}"
    !PYTHONPATH=/content/Siformer python -m train --experiment_name {name} --num_classes 100 --num_worker 2 --seed 0 \
      --training_set_path "{CSV}/train_smote.csv" --validation_set from-file --validation_set_path "{CSV}/val.csv" \
      --testing_set_path "{CSV}/test.csv" 2>&1 | tee {name}.log